## Setup

In [ ]:
import os
import io
import json
import zipfile
from datetime import datetime, date
import re

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import networkx as nx
import matplotlib.pyplot as plt
import contextily as ctx
import matplotlib.pyplot as plt

import zipfile
import numpy as np
from sklearn.cluster import DBSCAN
import folium

## Preprocessing

In [ ]:
# API_KEY = "DPSQebXKPfwlHqcvc3tHX5DcD533FNFg"
# BASE_URL = "https://api.transit.land/api/v2/rest"

# FEED_ONESTOP_ID = "f-9q5-metro~losangeles~rail"   # LA Metro Rail feed
# YEARS = list(range(2020, 2026))

In [ ]:
# def get_midyear_feed_version(feed_onestop_id, year, api_key):
#     """
#     Fetch the single GTFS feed_version active close to midyear for a given year.
#     Uses service_date filtering to minimize API calls.

#     Attempts several dates around June 30 to maximize success.
#     """
#     base_url = f"{BASE_URL}/feed_versions"

#     candidate_dates = [
#         f"{year}-06-30",
#         f"{year}-06-15",
#         f"{year}-07-15",
#         f"{year}-06-01",
#         f"{year}-07-01",
#     ]

#     for d in candidate_dates:
#         params = {
#             "feed_onestop_id": feed_onestop_id,
#             "service_date": d,
#             "limit": 1,
#             "api_key": api_key
#         }

#         r = requests.get(base_url, params=params)
#         if r.status_code != 200:
#             continue

#         feed_versions = r.json().get("feed_versions", [])
#         if len(feed_versions) > 0:
#             print(f"✔ Found feed version for {year} using service_date {d}")
#             return feed_versions[0]

#     print(f"✖ No feed version found for {year}")
#     return None

In [ ]:
# chosen_versions = {}

# for y in YEARS:
#     fv = get_midyear_feed_version(FEED_ONESTOP_ID, y, API_KEY)
#     chosen_versions[y] = fv

# chosen_versions

✔ Found feed version for 2020 using service_date 2020-06-30
✔ Found feed version for 2021 using service_date 2021-06-30
✔ Found feed version for 2022 using service_date 2022-06-30
✔ Found feed version for 2023 using service_date 2023-06-30
✔ Found feed version for 2024 using service_date 2024-06-30
✔ Found feed version for 2025 using service_date 2025-06-30


{2020: {'earliest_calendar_date': '2025-12-10',
  'feed': {'name': None,
   'onestop_id': 'f-9q5-metro~losangeles~rail',
   'spec': 'GTFS'},
  'feed_infos': [],
  'feed_version_gtfs_import': None,
  'fetched_at': '2025-12-11T01:17:10.05043Z',
  'files': [{'csv_like': True,
    'header': 'agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone',
    'name': 'agency.txt',
    'rows': 1,
    'sha1': 'bea8d585839c3d48808d994cb064468e66ca76d6',
    'size': 174},
   {'csv_like': True,
    'header': 'service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date',
    'name': 'calendar.txt',
    'rows': 23,
    'sha1': 'b1f09d944998f3e4603082d6a15a1ed2d3f67ae9',
    'size': 1404},
   {'csv_like': True,
    'header': 'service_id,date,exception_type',
    'name': 'calendar_dates.txt',
    'rows': 0,
    'sha1': '8b72d0b8397c7832d9074a7ee8e99b8a2d76ad62',
    'size': 32},
   {'csv_like': True,
    'header': 'fare_id,price,currency_type,payment_method,tr

In [ ]:
# def download_gtfs_from_feed_version(feed_version, year, dest_dir="gtfs_la"):
#     if feed_version is None:
#         return None

#     os.makedirs(dest_dir, exist_ok=True)

#     url = feed_version["url"]
#     sha = feed_version["sha1"]
#     path = os.path.join(dest_dir, f"la_rail_{year}_{sha}.zip")  # ← include year

#     print(f"⬇ Downloading GTFS for {year} (sha {sha})...")
#     r = requests.get(url)
#     r.raise_for_status()

#     with open(path, "wb") as f:
#         f.write(r.content)

#     print(f"Saved to {path}")
#     return path
# gtfs_paths = {}

# for y, fv in chosen_versions.items():
#     gtfs_paths[y] = download_gtfs_from_feed_version(fv, y)


⬇ Downloading GTFS for 2020 (sha 6f50aa4673881b3a739cce6a10e977da3bbca2b1)...
Saved to gtfs_la/la_rail_2020_6f50aa4673881b3a739cce6a10e977da3bbca2b1.zip
⬇ Downloading GTFS for 2021 (sha 6f50aa4673881b3a739cce6a10e977da3bbca2b1)...
Saved to gtfs_la/la_rail_2021_6f50aa4673881b3a739cce6a10e977da3bbca2b1.zip
⬇ Downloading GTFS for 2022 (sha 6f50aa4673881b3a739cce6a10e977da3bbca2b1)...
Saved to gtfs_la/la_rail_2022_6f50aa4673881b3a739cce6a10e977da3bbca2b1.zip
⬇ Downloading GTFS for 2023 (sha 6f50aa4673881b3a739cce6a10e977da3bbca2b1)...
Saved to gtfs_la/la_rail_2023_6f50aa4673881b3a739cce6a10e977da3bbca2b1.zip
⬇ Downloading GTFS for 2024 (sha 6f50aa4673881b3a739cce6a10e977da3bbca2b1)...
Saved to gtfs_la/la_rail_2024_6f50aa4673881b3a739cce6a10e977da3bbca2b1.zip
⬇ Downloading GTFS for 2025 (sha 6f50aa4673881b3a739cce6a10e977da3bbca2b1)...
Saved to gtfs_la/la_rail_2025_6f50aa4673881b3a739cce6a10e977da3bbca2b1.zip


## Network Creation & Visualization

In [ ]:
def load_gtfs(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        def read(name):
            return pd.read_csv(z.open(name))
        return (
            read("stops.txt"),
            read("routes.txt"),
            read("trips.txt"),
            read("stop_times.txt")
        )

def build_rail_graph(gtfs_zip_path):
    with zipfile.ZipFile(gtfs_zip_path, "r") as z:
        stops = pd.read_csv(z.open("stops.txt"))
        routes = pd.read_csv(z.open("routes.txt"))
        trips = pd.read_csv(z.open("trips.txt"))
        stop_times = pd.read_csv(z.open("stop_times.txt"))

    # Clean
    stops["stop_lat"] = stops["stop_lat"].astype(float)
    stops["stop_lon"] = stops["stop_lon"].astype(float)
    stops["stop_id"] = stops["stop_id"].astype(str)
    stop_times["stop_id"] = stop_times["stop_id"].astype(str)

    # -------------------------
    # Map stop_id → station_id
    # -------------------------
    stop_to_station = {}

    for _, r in stops.iterrows():
        lt = r.get("location_type", 0)

        if lt == 1:
            # real station
            stop_to_station[r["stop_id"]] = r["stop_id"]

        elif lt == 0:
            # platform → map to parent_station
            stop_to_station[r["stop_id"]] = r["parent_station"]

        # ignore entrances (2)

    # Apply mapping
    stop_times["station_id"] = stop_times["stop_id"].map(stop_to_station)
    stop_times = stop_times.dropna(subset=["station_id"])

    # -------------------------
    # Filter to rail routes
    # -------------------------
    rail_routes = routes[routes["route_type"].isin([0, 1, 2])]
    rail_trips = trips[trips["route_id"].isin(rail_routes["route_id"])]
    st = stop_times[stop_times["trip_id"].isin(rail_trips["trip_id"])].copy()

    # -------------------------
    # Build station graph
    # -------------------------
    G = nx.DiGraph()

    # Add station nodes
    station_rows = stops[stops["location_type"] == 1]

    for _, r in station_rows.iterrows():
        sid = r["stop_id"]  # station id ends with S
        G.add_node(
            sid,
            name=r["stop_name"],
            lat=r["stop_lat"],
            lon=r["stop_lon"]
        )

    # Add edges between consecutive stations
    for trip_id, group in st.groupby("trip_id"):
        g = group.sort_values("stop_sequence").reset_index(drop=True)

        for i in range(len(g) - 1):
            u = g.loc[i, "station_id"]
            v = g.loc[i + 1, "station_id"]

            if u != v and u in G.nodes and v in G.nodes:
                G.add_edge(u, v)

    return G

def graph_to_geodataframes(G):
    # Nodes
    nodes = []
    for nid, data in G.nodes(data=True):
        nodes.append({
            "stop_id": nid,
            "name": data.get("name"),
            "parent": data.get("parent_station"),
            "geometry": Point(data["lon"], data["lat"])
        })
    gdf_nodes = gpd.GeoDataFrame(nodes, crs="EPSG:4326")

    # Edges
    edges = []
    for u, v, data in G.edges(data=True):
        pu = Point(G.nodes[u]["lon"], G.nodes[u]["lat"])
        pv = Point(G.nodes[v]["lon"], G.nodes[v]["lat"])
        edges.append({
            "u": u,
            "v": v,
            "routes": list(data.get("routes", [])),
            "geometry": LineString([pu, pv])
        })
    gdf_edges = gpd.GeoDataFrame(edges, crs="EPSG:4326")

    return gdf_nodes, gdf_edges


In [ ]:
def plot_rail_line(G, zoom_start=11):
    # Convert NetworkX → GeoDataFrames
    gdf_nodes, gdf_edges = graph_to_geodataframes(G)

    # Center map on average location
    center_lat = gdf_nodes.geometry.y.mean()
    center_lon = gdf_nodes.geometry.x.mean()

    # Create folium map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    # --- Draw edges as polylines ---
    for _, row in gdf_edges.iterrows():
        coords = [(pt[1], pt[0]) for pt in row.geometry.coords] # Fixed: Changed pt.y to pt[1] and pt.x to pt[0]
        folium.PolyLine(
            locations=coords,
            color="blue",
            weight=3,
            opacity=0.7,
            tooltip=f"{row['u']} → {row['v']} (routes: {row['routes']})"
        ).add_to(m)

    # --- Draw nodes as circles ---
    for _, row in gdf_nodes.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4,
            color="red",
            fill=True,
            fill_color="red",
            popup=(
                f"<b>Stop ID:</b> {row['stop_id']}<br>"
                f"<b>Name:</b> {row['name']}<br>"
                f"<b>Parent Station:</b> {row['parent']}"
            )
        ).add_to(m)

    return m

In [ ]:
# def get_gtfs_paths_manual(folder_path="gtfs_la"):
#     gtfs_paths = {}
#     for filename in os.listdir(folder_path):
#         if filename.endswith(".zip"):
#             match = re.search(r'(\d{4})', filename)  # Extract 4-digit year
#             if match:
#                 year = int(match.group(0))
#                 full_path = os.path.join(folder_path, filename)
#                 gtfs_paths[year] = full_path
#     return gtfs_paths

# gtfs_paths_manual = get_gtfs_paths_manual()
# print("Manually collected GTFS paths:")
# print(gtfs_paths_manual)

In [ ]:
graphs = {}
nodes_gdf = {}
edges_gdf = {}

gtfs_paths = {
    2016: "la_rail_2016_f-9q5-metro~losangeles~rail-91f343f0efbb23dd02fa68c623288278e83990a5.zip",
    2017: "la_rail_2017_f-9q5-metro~losangeles~rail-e281153e74ef7c2c189fc047f85a278ffab5fbe3.zip",
    2018: "la_rail_2018_f-9q5-metro~losangeles~rail-a4a86ee18343f82fc46a59301ecb87e39c45f1dc.zip",
    2019: "la_rail_2019_f-9q5-metro~losangeles~rail-981bf570448cc88e1ffcccf9305acc2b660e4a00.zip",
    2020: "la_rail_2020_f-9q5-metro~losangeles~rail-d349dd6eb59a9b87bd49291761ba820beff65d98.zip",
    2021: "la_rail_2021_f-9q5-metro~losangeles~rail-9597b18e0cdd0a58c884c25d41450cf199571e72.zip",
    2022: "la_rail_2022_f-9q5-metro~losangeles~rail-c69e240ac6eb290c117ea88934934e7871fd702c.zip",
    2023: "la_rail_2023_f-9q5-metro~losangeles~rail-da0935270e8394b17d7dfe72528b175cc377ef7d.zip"
}

# with zipfile.ZipFile(gtfs_paths[2016], "r") as z:
#     stops = pd.read_csv(z.open("stops.txt"))
# stops.head()

for year, path in gtfs_paths.items():
    if path is None:
        print(f"No GTFS for {year}")
        continue

    print(f"\nBuilding graph for {year} ...")

    try:
        G = build_rail_graph(path)
        graphs[year] = G

        g_nodes, g_edges = graph_to_geodataframes(G)
        nodes_gdf[year] = g_nodes
        edges_gdf[year] = g_edges

        print(f"✓ Successfully built graph for {year} with {len(G.nodes())} nodes and {len(G.edges())} edges.")

    except Exception as e:
        print(f"❌ Error building graph for {year}: {e}")


Building graph for 2016 ...
✓ Successfully built graph for 2016 with 93 nodes and 181 edges.

Building graph for 2017 ...
✓ Successfully built graph for 2017 with 93 nodes and 181 edges.

Building graph for 2018 ...
✓ Successfully built graph for 2018 with 93 nodes and 181 edges.

Building graph for 2019 ...
✓ Successfully built graph for 2019 with 93 nodes and 161 edges.

Building graph for 2020 ...
✓ Successfully built graph for 2020 with 93 nodes and 181 edges.

Building graph for 2021 ...
✓ Successfully built graph for 2021 with 93 nodes and 177 edges.

Building graph for 2022 ...
✓ Successfully built graph for 2022 with 93 nodes and 177 edges.

Building graph for 2023 ...
✓ Successfully built graph for 2023 with 104 nodes and 199 edges.


In [ ]:
# Display the plot for a specific year, for example, 2020
# plot_rail_line(graphs[2025])

# os.makedirs("la_rail_gpkg", exist_ok=True)

# for y in graphs:
#     out = f"la_rail_gpkg/la_rail_{y}.gpkg"
#     nodes_gdf[y].to_file(out, layer="nodes", driver="GPKG")
#     edges_gdf[y].to_file(out, layer="edges", driver="GPKG")
#     print("Saved:", out)


In [ ]:
def graph_nodes_gdf(G):
    rows = []
    for nid, data in G.nodes(data=True):
        rows.append({
            "stop_id": nid,
            "name": data.get("name"),
            "parent": data.get("parent_station"),
            "lat": data.get("lat"),
            "lon": data.get("lon"),
            "geometry": Point(data["lon"], data["lat"]),
        })
    gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326")
    return gdf

def station_diffs(graphs, year):
    years = sorted(graphs.keys())
    idx = years.index(year)

    # First year → nothing is added/removed relative to earlier data
    if idx == 0:
        return set(), set()

    prev_year = years[idx - 1]
    G_prev = graphs[prev_year]
    G_curr = graphs[year]

    prev_nodes = set(G_prev.nodes())
    curr_nodes = set(G_curr.nodes())

    added = curr_nodes - prev_nodes
    removed = prev_nodes - curr_nodes

    return added, removed

import folium

def edge_diffs(graphs, year):
    years = sorted(graphs.keys())
    idx = years.index(year)

    if idx == 0:
        return set(), set()

    prev_year = years[idx - 1]
    G_prev = graphs[prev_year]
    G_curr = graphs[year]

    # Extract simple directed edges (no metadata)
    prev_edges = set((u, v) for u, v in G_prev.edges())
    curr_edges = set((u, v) for u, v in G_curr.edges())

    added_edges = curr_edges - prev_edges
    removed_edges = prev_edges - curr_edges

    return added_edges, removed_edges

def render_map_year(graphs, year):
    G = graphs[year]

    # Compute historical diffs
    added_nodes, removed_nodes = station_diffs(graphs, year)
    added_edges, removed_edges = edge_diffs(graphs, year)

    # Convert G → GeoDataFrames
    gdf_nodes = graph_nodes_gdf(G)
    _, gdf_edges = graph_to_geodataframes(G)

    # Center map
    center_lat = gdf_nodes.geometry.y.mean()
    center_lon = gdf_nodes.geometry.x.mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

    # --- DRAW EDGES WITH HISTORICAL COLORING ---
    for _, row in gdf_edges.iterrows():
        u = row["u"]
        v = row["v"]

        coords = [(pt[1], pt[0]) for pt in row.geometry.coords]

        if (u, v) in added_edges:
            color = "green"
            weight = 5
        elif (u, v) in removed_edges:
            color = "red"
            weight = 5
        else:
            color = "blue"
            weight = 3

        folium.PolyLine(
            coords,
            color=color,
            weight=weight,
            opacity=0.8
        ).add_to(m)

    # --- DRAW NODES ---
    for _, row in gdf_nodes.iterrows():
        sid = row["stop_id"]

        if sid in added_nodes:
            color = "green"
        elif sid in removed_nodes:
            color = "red"
        else:
            color = "black"

        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=1.0,
            popup=(f"<b>{row['stop_id']}</b><br>{row['name']}")
        ).add_to(m)

    return m


In [ ]:
import ipywidgets as widgets
from IPython.display import display

years_sorted = sorted(graphs.keys())

year_slider = widgets.IntSlider(
    value=years_sorted[0],
    min=years_sorted[0],
    max=years_sorted[-1],
    step=1,
    description="Year:"
)

map_output = widgets.Output()

def update_map(change=None):
    year = year_slider.value
    with map_output:
        map_output.clear_output(wait=True);
        display(render_map_year(graphs, year))

year_slider.observe(update_map, names='value')

display(year_slider, map_output)

# Initial draw
update_map()


IntSlider(value=2016, description='Year:', max=2023, min=2016)

Output()

In [ ]:
def compare_station_names(graphs, year1, year2):
    G1 = graphs[year1]
    G2 = graphs[year2]

    names1 = {data["name"] for _, data in G1.nodes(data=True)}
    names2 = {data["name"] for _, data in G2.nodes(data=True)}

    added = names2 - names1
    removed = names1 - names2

    print("ADDED:", added)
    print("REMOVED:", removed)
compare_station_names(graphs, 2022, 2023)

ADDED: {'Grand Ave Arts / Bunker Hill Station', 'Leimert Park Station', 'Fairview Heights Station', 'Downtown Inglewood Station', 'Westchester / Veterans Station', 'Historic Broadway Station', 'Expo / Crenshaw K-Line Station', 'Hyde Park Station', 'Martin Luther King Jr Station', 'Aviation / Century Station', 'Airport Metro Connector', 'Expo / Crenshaw E-Line Station'}
REMOVED: {'Expo / Crenshaw Station'}


In [ ]:
print(len(graphs[2022].nodes))
print(len(graphs[2023].edges))

93
199


## Station-Level Centralities

In [ ]:
def eigen_or_katz(G, weight="weight", alpha=0.01, beta=1.0):
    """
    Compute eigenvector centrality per connected component.
    If eigenvector fails, fall back to Katz centrality.
    """

    eigen_all = {}

    for component in nx.connected_components(G):
        subG = G.subgraph(component)

        # Try eigenvector centrality
        try:
            eig = nx.eigenvector_centrality(subG, weight=weight, max_iter=1000)
            eigen_all.update(eig)
            continue

        except nx.PowerIterationFailedConvergence:
            print(f"  ⚠️ Eigenvector failed for component of size {len(subG)} — switching to Katz.")

        # If eigenvector fails, use Katz centrality
        try:
            katz = nx.katz_centrality(subG, alpha=alpha, beta=beta, weight=weight, max_iter=1000)
            eigen_all.update(katz)
        except Exception as e:
            print(f"  ❌ Katz also failed: {e}")
            # Last fallback — assign zeros
            eigen_all.update({node: 0 for node in subG.nodes()})

    return eigen_all

In [ ]:
def compute_station_centralities(G, weight="weight"):
    if G.is_directed():
        G = G.to_undirected()

    # Centralities
    degree_c = nx.degree_centrality(G)
    betweenness_c = nx.betweenness_centrality(G, weight=weight, normalized=True)
    closeness_c = nx.closeness_centrality(G, distance=weight)
    eigen_c = eigen_or_katz(G, weight=weight)
    pagerank_c = nx.pagerank(G, weight=weight)

    # Store results in rows (this allows merging metadata cleanly)
    rows = []
    for node, data in G.nodes(data=True):

        rows.append({
            "station_id": node,

            # station metadata (ADDED)
            "station_name": data.get("name"),
            "lat": data.get("lat"),
            "lon": data.get("lon"),

            # centralities
            "degree": degree_c[node],
            "betweenness": betweenness_c[node],
            "closeness": closeness_c[node],
            "eigen_or_katz": eigen_c[node],
            "pagerank": pagerank_c[node]
        })

    df = pd.DataFrame(rows).set_index("station_id")
    return df

In [ ]:
def compute_centralities_all_years(graphs, weight="weight"):
    results = []

    for year, G in graphs.items():
        print(f"Computing centralities for {year}...")

        df = compute_station_centralities(G, weight=weight)
        df["year"] = year
        df["station_id"] = df.index

        results.append(df)

    centrality_panel = pd.concat(results, ignore_index=True)
    return centrality_panel

In [ ]:
centrality_df = compute_centralities_all_years(graphs)

Computing centralities for 2020...
Computing centralities for 2021...
Computing centralities for 2022...
Computing centralities for 2023...
Computing centralities for 2024...
Computing centralities for 2025...


In [ ]:
import ipywidgets as widgets
from IPython.display import display

def interactive_station_comparison(centrality_df, label_attr="name"):
    """
    Interactive dropdown to compare station centralities across years.
    centrality_df must contain columns: station_id, year, degree, betweenness, closeness, eigen_or_katz, pagerank
    """

    # Build dictionary: station label → id
    station_options = {}

    for station_id in centrality_df["station_id"].unique():
        # Try to get station name from your station attributes stored in graphs
        # If graphs aren't available here, fallback to station_id string
        name = None
        for G in graphs.values():
            if station_id in G.nodes:
                name = G.nodes[station_id].get("name", None)
                break

        label = name if name else str(station_id)
        station_options[label] = station_id

    dropdown = widgets.Dropdown(
        options=station_options,
        description="Station:",
        layout=widgets.Layout(width="350px"),
        style={"description_width": "80px"}
    )

    output = widgets.Output()

    def on_change(change):
        if change["type"] == "change" and change["name"] == "value":
            station = change["new"]
            with output:
                output.clear_output()

                df = (
                    centrality_df[centrality_df.station_id == station]
                    .sort_values("year")
                    .set_index("year")
                )

                print(f"Centrality values for station: {station}\n")
                display(df)

    dropdown.observe(on_change)
    display(dropdown, output)

# Usage:
interactive_station_comparison(centrality_df)

Dropdown(description='Station:', layout=Layout(width='350px'), options={'Downtown Long Beach Station': '80101S…

Output()

In [ ]:
centrality_df.to_csv("station_centrality_panel.csv", index=False)

## Census-Level Centralities

In [ ]:
import requests
import time

def latlon_to_tract(lat, lon):
    """
    Convert latitude/longitude to Census Tract GEOID using the Census Geocoder API.
    Returns None if lookup fails.
    """
    url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
    params = {
        "x": lon,
        "y": lat,
        "benchmark": "2020",
        "vintage": "2020",
        "format": "json",
    }

    try:
        r = requests.get(url, params=params, timeout=5).json()
        tract = r["result"]["geographies"]["Census Tracts"][0]["GEOID"]
        return tract
    except Exception as e:
        return None


def assign_tracts(df):
    """
    Add tract_geoid column to the centrality dataframe.
    """
    tracts = []

    for idx, row in df.iterrows():
        tract = latlon_to_tract(row["lat"], row["lon"])
        tracts.append(tract)
        time.sleep(0.05)  # avoid API throttling

    df["tract_geoid"] = tracts
    return df


# ------------------------------
# RUN TRACT ASSIGNMENT
# ------------------------------

centrality_df = assign_tracts(centrality_df)

# Now add the census full summary-level code
centrality_df["census_full_code"] = (
    "1400000US" + centrality_df["tract_geoid"].astype(str)
)

# Save
centrality_df.to_csv("station_centrality_with_tracts.csv", index=False)

In [ ]:
centrality_df["is_LA_county"] = centrality_df["tract_geoid"].astype(str).str.startswith("06037")
centrality_df["is_LA_county"].value_counts()

,count
is_LA_county,
True,591


In [ ]:
# -----------------------------------
# Ensure tract_geoid is stored as a string
# -----------------------------------
centrality_df["tract_geoid"] = centrality_df["tract_geoid"].astype(str)

# Add the full Census Summary Level code (kept as string)
centrality_df["census_full_code"] = "1400000US" + centrality_df["tract_geoid"].astype(str)


# -----------------------------------
# TRACT-LEVEL AGGREGATIONS
# -----------------------------------

tract_df_sum = centrality_df.groupby(["year", "tract_geoid", "census_full_code"]).agg({
    "degree": "sum",
    "betweenness": "sum",
    "closeness": "sum",
    "eigen_or_katz": "sum",
    "pagerank": "sum"
}).reset_index()


tract_df_mean = centrality_df.groupby(["year", "tract_geoid", "census_full_code"]).agg({
    "degree": "mean",
    "betweenness": "mean",
    "closeness": "mean",
    "eigen_or_katz": "mean",
    "pagerank": "mean"
}).reset_index()


tract_df_max = centrality_df.groupby(["year", "tract_geoid", "census_full_code"]).agg({
    "degree": "max",
    "betweenness": "max",
    "closeness": "max",
    "eigen_or_katz": "max",
    "pagerank": "max"
}).reset_index()


# -----------------------------------
# MERGE THE THREE AGGREGATION TABLES
# -----------------------------------

tract_df = tract_df_sum.merge(
    tract_df_mean,
    on=["tract_geoid", "census_full_code", "year"],
    suffixes=("_sum", "_mean")
).merge(
    tract_df_max,
    on=["tract_geoid", "census_full_code", "year"],
    suffixes=("", "_max")
)


# -----------------------------------
# Final ensure GEOID formats are preserved as strings
# -----------------------------------
tract_df["tract_geoid"] = tract_df["tract_geoid"].astype(str)
tract_df["census_full_code"] = tract_df["census_full_code"].astype(str)


# -----------------------------------
# SAVE TO CSV
# -----------------------------------
tract_df.to_csv("tract_centrality_panel.csv", index=False)

## Tract Panels

In [ ]:
import pandas as pd
import re
import os

def load_acs_file(filepath):
    """
    Loads an ACS CSV in your format:
    - Keeps all Estimate + MoE columns
    - Removes metadata row (row 0)
    - Extracts tract_geoid
    - Extracts year from filename
    """
    # Extract year from filename
    filename = os.path.basename(filepath)
    year_match = re.search(r"(\d{4})", filename)
    if not year_match:
        raise ValueError(f"Could not find year in filename: {filename}")
    year = int(year_match.group(1))

    # Load CSV
    df = pd.read_csv(filepath)

    # Remove metadata row ("Estimate", "MoE")
    df = df.iloc[1:].reset_index(drop=True)

    # Extract tract_geoid
    df["tract_geoid"] = df["Geography"].str.replace("1400000US", "", regex=False)
    df["tract_geoid"] = df["tract_geoid"].astype(str)

    # Add year column
    df["year"] = year

    return df

In [ ]:
rent_2020 = load_acs_file("Gross Rent as a Percentage of Household Income (LA) - 2020.csv")
rent_2021 = load_acs_file("Gross Rent as a Percentage of Household Income (LA) - 2021.csv")
rent_2022 = load_acs_file("Gross Rent as a Percentage of Household Income (LA) - 2022.csv")
rent_2023 = load_acs_file("Gross Rent as a Percentage of Household Income (LA) - 2023.csv")
mortgage_2020 = load_acs_file("Mortgage Status by Median Value (Dollars) (LA) - 2020.csv")
mortgage_2021 = load_acs_file("Mortgage Status by Median Value (Dollars) (LA) - 2021.csv")
mortgage_2022 = load_acs_file("Mortgage Status by Median Value (Dollars) (LA) - 2022.csv")
mortgage_2023 = load_acs_file("Mortgage Status by Median Value (Dollars) (LA) - 2023.csv")
housingunits_2020 = load_acs_file("Housing Units (LA) - 2020.csv")
housingunits_2021 = load_acs_file("Housing Units (LA) - 2021.csv")
housingunits_2022 = load_acs_file("Housing Units (LA) - 2022.csv")
housingunits_2023 = load_acs_file("Housing Units (LA) - 2023.csv")

In [ ]:
rent_panel = pd.concat([rent_2020, rent_2021, rent_2022, rent_2023], ignore_index=True)
mortgage_panel = pd.concat([mortgage_2020, mortgage_2021, mortgage_2022, mortgage_2023], ignore_index=True)
housingunits_panel = pd.concat([housingunits_2020, housingunits_2021, housingunits_2022, housingunits_2023], ignore_index=True)

final_panel = (
    tract_df
    .merge(rent_panel, on=["tract_geoid", "year"], how="left")
    .merge(mortgage_panel, on=["tract_geoid", "year"], how="left")
    .merge(housingunits_panel, on=["tract_geoid", "year"], how="left")
)

In [ ]:
final_panel.to_csv("final_tract_panel_with_ACS.csv", index=False)

In [ ]:
def clean_final_panel(df):
    # Drop geography + non-useful text columns
    drop_cols = [col for col in df.columns
                 if col.startswith("Geography")
                 or col.startswith("Census Tract")
                 or col.lower().startswith("unnamed")
                 or col.lower() == "unnamed"]

    df = df.drop(columns=drop_cols, errors="ignore")

    # Drop duplicate centrality columns (from max merge)
    duplicates = ["degree", "betweenness", "closeness", "eigen_or_katz", "pagerank"]
    df = df.drop(columns=duplicates, errors="ignore")

    return df

cleaned_panel = clean_final_panel(final_panel)

ordered_cols = [
    "year",
    "tract_geoid",
    "census_full_code",

    # centrality (sum)
    "degree_sum", "betweenness_sum", "closeness_sum",
    "eigen_or_katz_sum", "pagerank_sum",

    # centrality (mean)
    "degree_mean", "betweenness_mean", "closeness_mean",
    "eigen_or_katz_mean", "pagerank_mean",
] + [col for col in cleaned_panel.columns if col not in
     ["year","tract_geoid","census_full_code",
      "degree_sum","betweenness_sum","closeness_sum",
      "eigen_or_katz_sum","pagerank_sum",
      "degree_mean","betweenness_mean","closeness_mean",
      "eigen_or_katz_mean","pagerank_mean"]]

cleaned_panel = cleaned_panel[ordered_cols]

cleaned_panel.to_csv("tract_panel_clean.csv", index=False)

## DiD Test

In [ ]:
panel = cleaned_panel.rename(columns={
    "50% or More": "rent50_est",
    "50% or More.1": "rent50_moe"
})
panel["rent50_est"] = pd.to_numeric(panel["rent50_est"], errors="coerce")

In [ ]:
panel = panel.sort_values(["tract_geoid", "year"])
panel["betweenness_sum_change"] = panel.groupby("tract_geoid")["betweenness_sum"].diff()

In [ ]:
panel["treated"] = (panel["betweenness_sum_change"] > 0).astype(int)

In [ ]:
first_change_year = int(panel.loc[
    panel["betweenness_sum_change"] > 0, "year"
].min())

panel["post"] = (panel["year"] >= first_change_year).astype(int)

In [ ]:
panel["did"] = panel["treated"] * panel["post"]

In [ ]:
from linearmodels.panel import PanelOLS

panel_for_reg = panel.set_index(["tract_geoid", "year"])

In [ ]:
mod = PanelOLS(
    dependent=panel_for_reg["rent50_est"],
    exog=panel_for_reg[["did"]],
    entity_effects=True,   # tract FE
    time_effects=True      # year FE
)

res = mod.fit(cov_type="robust")
print(res.summary)

/usr/local/lib/python3.12/dist-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                          PanelOLS Estimation Summary                           
Dep. Variable:             rent50_est   R-squared:                        0.0001
Estimator:                   PanelOLS   R-squared (Between):             -0.0003
No. Observations:                 344   R-squared (Within):              -0.0006
Date:                Thu, Dec 11 2025   R-squared (Overall):             -0.0003
Time:                        20:10:15   Log-likelihood                   -1784.9
Cov. Estimator:                Robust                                           
                                        F-statistic:                      0.0248
Entities:                          93   P-value                           0.8749
Avg Obs:                       3.6989   Distribution:                   F(1,247)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             0.0147
                            

## Archive

In [ ]:
yr = 2022

nodes = nodes_gdf[yr].to_crs(3857)
edges = edges_gdf[yr].to_crs(3857)

fig, ax = plt.subplots(figsize=(8,8))
edges.plot(ax=ax, linewidth=1, color="blue")
nodes.plot(ax=ax, markersize=12, color="red")

ctx.add_basemap(ax)
ax.set_title(f"LA Metro Rail – {yr}")
plt.show()

KeyError: 2022